# Component 2 — matched ordinal tooth-crop experiment (v2)

This experiment compares a weighted cross-entropy ResNet-18 crop classifier with a rank-consistent ordinal CORAL-style head. It reuses the same 680 Training images, five saved folds, source labels, preprocessing, and fold-specific YOLO detectors from v1. Only the crop-classifier objective/head changes.

The study labels are provisional. This notebook does not use the 53 Validation images for model selection, the 181 held records, or any protected Test data. It does not retrain YOLO, replace the live PP1 model, or establish clinical performance.

**Run with a CUDA GPU runtime. T4 is the intended Colab choice. CPU execution is blocked.** The first run may download the official torchvision ResNet-18 ImageNet V1 weights into Colab; their SHA-256 is checked before fitting.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_CANDIDATES = (
    Path('/content/drive/MyDrive/RP-J26-IT-458/roi_tooth_v1'),
    Path('/content/drive/MyDrive/roi_tooth_v1'),
)
INPUT_ZIP_NAMES = (
    'roi_tooth_dataset_v1_oneclass_v4.zip',
    'roi_tooth_v1_colab_return.zip',
    'roi_tooth_v2_coral_code_v1.zip',
)
DRIVE_ROOT = next(
    (folder for folder in DRIVE_CANDIDATES if all((folder / name).is_file() for name in INPUT_ZIP_NAMES)),
    None,
)
assert DRIVE_ROOT is not None, (
    'Could not find all three ZIPs. Put them in My Drive/RP-J26-IT-458/roi_tooth_v1 '
    'or in the shared-folder shortcut My Drive/roi_tooth_v1.'
)
PACKAGE_ZIP = DRIVE_ROOT / INPUT_ZIP_NAMES[0]
V1_RETURN_ZIP = DRIVE_ROOT / INPUT_ZIP_NAMES[1]
CODE_ZIP = DRIVE_ROOT / INPUT_ZIP_NAMES[2]
OUTPUT_ROOT = DRIVE_ROOT / 'roi_tooth_v2_coral_colab_run_v1'
V2_RETURN_ZIP = DRIVE_ROOT / 'roi_tooth_v2_coral_return.zip'

EXPECTED_PACKAGE_SHA256 = '9b1e797f7f593438a5318410ed2aa1cfdd7a66b8db0aac865b3229b63d3a20c8'
EXPECTED_V1_RETURN_SHA256 = 'ebb41badf4bd2154019dad55ba2f459ad1807c7f42b0722c13d849be2460f9bb'
EXPECTED_CODE_ZIP_SHA256 = '8af0d3596d3afe4586d5407ecedb27754315f606b7c0e8e55c8a39e08abfffdf'
EXPECTED_IMAGE_MANIFEST_SHA256 = '975f4cde460b69c074b2655910f597623454b774dc27768d75f8002ce2ff1619'
EXPECTED_PROTOCOL_SHA256 = '2253db0e89aad06a2fe49a90d313b7ae0e08a806b87845d49753b3d9e48dbc4b'
EXPECTED_FOLD_SHA256 = 'e202132b8b7ee38ec4041ea399bccb0b538ef75ce1861faec044292c11cf90d4'

print('Dataset:', PACKAGE_ZIP)
print('Frozen v1 detector bundle:', V1_RETURN_ZIP)
print('New source bundle:', CODE_ZIP)
print('New resumable output folder:', OUTPUT_ROOT)
print('New return ZIP:', V2_RETURN_ZIP)

## Verify and extract the three inputs

This cell checks the exact ZIP hashes and CRCs before extraction. It also rejects archive paths that could escape the extraction folder. Existing files are left in Drive unchanged.

In [ ]:
import hashlib
import json
import zipfile
from pathlib import PurePosixPath


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        bad_member = archive.testzip()
        assert bad_member is None, f'ZIP CRC failed at {bad_member}'
        for member in archive.namelist():
            member_path = PurePosixPath(member)
            assert not member_path.is_absolute() and '..' not in member_path.parts, f'Unsafe ZIP path: {member}'
        archive.extractall(destination)

for path, expected in (
    (PACKAGE_ZIP, EXPECTED_PACKAGE_SHA256),
    (V1_RETURN_ZIP, EXPECTED_V1_RETURN_SHA256),
    (CODE_ZIP, EXPECTED_CODE_ZIP_SHA256),
):
    assert path.is_file(), f'Missing input: {path}'
    actual = sha256_file(path)
    print(path.name, 'SHA-256:', actual)
    assert actual == expected, f'Unexpected input hash for {path.name}'

PACKAGE_EXTRACT = Path('/content/roi_tooth_v2_package')
V1_RETURN_EXTRACT = Path('/content/roi_tooth_v1_return')
CODE_ROOT = Path('/content/roi_tooth_v2_code')
safe_extract(PACKAGE_ZIP, PACKAGE_EXTRACT)
safe_extract(V1_RETURN_ZIP, V1_RETURN_EXTRACT)
safe_extract(CODE_ZIP, CODE_ROOT)
PACKAGE_ROOT = PACKAGE_EXTRACT / 'roi_tooth_v1'
assert PACKAGE_ROOT.is_dir()
assert sha256_file(PACKAGE_ROOT / 'image_manifest.json') == EXPECTED_IMAGE_MANIFEST_SHA256

# The v1 ZIP wraps its files in a named folder; find that verified bundle root.
completion_files = list(V1_RETURN_EXTRACT.rglob('experiment_complete.json'))
assert len(completion_files) == 1, f'Expected one v1 completion record, found {len(completion_files)}'
V1_RETURN_ROOT = completion_files[0].parent

with (CODE_ROOT / 'code_manifest.json').open() as stream:
    code_manifest = json.load(stream)
for name, expected in code_manifest.items():
    path = CODE_ROOT / name
    assert path.is_file() and path.stat().st_size == expected['bytes']
    assert sha256_file(path) == expected['sha256'], f'Code file changed: {name}'

with (V1_RETURN_ROOT / 'experiment_complete.json').open() as stream:
    v1_complete = json.load(stream)
assert v1_complete['status'] == 'all_requested_folds_completed'
assert v1_complete['test_split_used'] is False
print('All three inputs passed ZIP CRC, SHA-256 and embedded-file checks.')

## Check the saved data protocol and GPU

The training function independently checks the frozen fold assignment, class mapping, fit/score membership, and detector checkpoint hashes. This cell confirms CUDA is active before installing any vision package.

In [ ]:
import torch
import sys

assert torch.cuda.is_available(), 'Stop: choose Runtime > Change runtime type > T4 GPU; CPU training is disabled.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

# Add the verified code bundle to Python's import path.
sys.path.insert(0, str(CODE_ROOT))

# Install the same pinned YOLO runtime used for v1. This does not upload data.
import subprocess
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PACKAGE_ROOT / 'code' / 'requirements-vision-roi-v1.txt'),
], check=True)

import ultralytics
import torchvision
print('Ultralytics:', ultralytics.__version__)
print('Torchvision:', torchvision.__version__)
assert ultralytics.__version__ == '8.4.178'

from roi_ordinal_v2 import _validate_data_protocol, _verify_detector_return
manifest, folds, crop_rows = _validate_data_protocol(PACKAGE_ROOT)
detector_hashes = _verify_detector_return(V1_RETURN_ROOT)
print('Allowlisted records:', len(manifest['records']))
print('Training images:', sum(row['original_split'] == 'Training' for row in manifest['records']))
print('Validation metadata records:', sum(row['original_split'] == 'Validation' for row in manifest['records']))
print('Fixed folds:', len(folds['folds']), 'fit:', len(folds['folds'][0]['fit_ids']), 'score:', len(folds['folds'][0]['score_ids']))
print('Training crop rows:', sum(row['source_split'] == 'Training' for row in crop_rows))
print('Verified fold detectors:', sorted(detector_hashes))

## Run the matched five-fold comparison

Each fold trains two crop classifiers on the same 544 fit images and 6,528 fit crops, then scores the same 136 score images and 1,632 source-annotation crops. The matching fold's YOLO detector is frozen and shared by both arms. Checkpoints save after every epoch to Drive and resume only when the saved configuration matches exactly.

The weighted-CE arm uses inverse-frequency weights calculated from its fit crops. The ordinal arm uses the CORAL-style ordered threshold head and equal threshold-loss weighting. No validation/test image influences training, tuning, or model selection.

In [ ]:
from roi_ordinal_v2 import run_experiment

RESULT = run_experiment(
    package_root=PACKAGE_ROOT,
    detector_root=V1_RETURN_ROOT,
    output_root=OUTPUT_ROOT,
    dataset_zip_sha256=EXPECTED_PACKAGE_SHA256,
    v1_return_zip_sha256=EXPECTED_V1_RETURN_SHA256,
    code_bundle_sha256=EXPECTED_CODE_ZIP_SHA256,
    device='cuda:0',
    epochs=12,
    batch_size=64,
    detector_batch=16,
    workers=2,
    seed=42,
    learning_rate=1e-4,
)
print('Run status:', RESULT['status'])
print('Output:', RESULT['output_root'])
print('Completion:', RESULT['completion'])

## Read the measured results

The primary comparison is image-level out-of-fold Macro-F1 across the four supported image grades, with accuracy, balanced accuracy, confusion matrix, per-grade recall, quadratic weighted kappa, and ordinal MAE reported as well. Crop-level metrics are reported separately. The detector's earlier mAP is unchanged because the detector is frozen; mAP is not a grade-classification metric.

The paired bootstrap describes fixed saved predictions only. It does not include refitting uncertainty or unknown participant overlap. Image-level AUROC/Brier are omitted because taking the maximum grade across detected teeth does not define a validated image-level probability.

In [ ]:
metrics_path = OUTPUT_ROOT / 'oof_metrics.json'
with metrics_path.open() as stream:
    metrics = json.load(stream)
assert metrics['status'] == 'complete_680_image_oof'
assert metrics['n_images_per_arm'] == {'weighted_ce': 680, 'coral': 680}
assert metrics['test_split_used'] is False

for arm in ('weighted_ce', 'coral'):
    item = metrics['metrics'][arm]
    print('\n', arm)
    print('Accuracy:', item.get('accuracy_all_images'))
    print('Macro-F1:', item.get('macro_f1_supported_grades_1_to_4'))
    print('Balanced accuracy:', item.get('balanced_accuracy_supported_grades_1_to_4'))
    print('Recall:', item.get('per_grade_recall'))
    print('QWK:', item.get('quadratic_weighted_kappa'))
    print('Ordinal MAE:', item.get('ordinal_mae'))
    print('Confusion matrix labels:', item.get('confusion_matrix_labels'))
    print('Confusion matrix:', item.get('confusion_matrix'))
print('Paired bootstrap:', metrics['paired_bootstrap_coral_minus_weighted_ce'])

## Package the new result for the Mac

This creates a new return ZIP containing the five new CORAL crop-classifier checkpoints, matching frozen detector copies, OOF evidence, metrics, configuration and integrity manifest. The weighted-CE checkpoints stay in the run folder on Drive for comparison. The v1 return ZIP and PP1 model remain untouched.

After this cell, download `roi_tooth_v2_coral_return.zip` from the Drive folder. On the Mac, place it under `data/vision/roi_tooth_v1_20261017/ordinal_v2/colab_return/` and verify its SHA-256 manifest before using any checkpoint. Do not report improvement until the results have been compared on the same 680 OOF records.

In [ ]:
from roi_ordinal_v2 import create_return_bundle

RETURN_INFO = create_return_bundle(OUTPUT_ROOT, V1_RETURN_ROOT, V2_RETURN_ZIP)
print(RETURN_INFO)
print('Return ZIP:', V2_RETURN_ZIP)
print('It is saved beside the verified v1 package in Google Drive.')

## Student explanation / viva

- `fit_ids` and `score_ids` come from the frozen five-fold file. A crop stays with its parent image, so no tooth from one photo crosses fit and score.
- Weighted cross-entropy predicts one of five crop grades. CORAL predicts four ordered threshold answers; grade 2 means above 0 and 1, but not above 2 or 3.
- YOLO is fixed. It finds tooth ROIs; each classifier grades those same boxes, and the photo grade is the maximum predicted tooth grade.
- We compare photo-level metrics separately from tooth-crop metrics. Detector mAP only describes box localization.
- A positive result would support this method on this provisional development cohort only. It would not verify clinical labels, establish participant independence, support whole-image MGI 0, or replace the final untouched Test evaluation.